In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

INPUT_TSV = "../data/sabdab_summary_all.tsv"
OUTPUT_FASTA = "../data/sabdab_h+l_chains_protein.fasta"
CHECKPOINT_DIR = Path("../data/checkpoints_h+l_chains_protein")
CHECKPOINT_INTERVAL = 100

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def get_fasta_chains(pdb_id):
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}"
    try:
        response = requests.get(url)
        if response.status_code != 200:
            return {}
        fasta_blocks = response.text.strip().split(">")
        chain_seqs = {}
        for block in fasta_blocks:
            lines = block.strip().splitlines()
            if not lines:
                continue
            header = lines[0]
            sequence = "".join(lines[1:])
            if "|Chain " in header or "|Chains " in header:
                chain_field = header.split("|")[1]
                parts = chain_field.replace("Chains ", "").replace("Chain ", "").split(",")
                for part in parts:
                    part = part.strip()
                    if "[auth " in part:
                        auth_id = part.split("[auth")[1].replace("]", "").strip()
                    else:
                        auth_id = part.strip()
                    chain_seqs[auth_id] = sequence
        return chain_seqs
    except Exception as e:
        print(f"[Error fetching {pdb_id}]: {e}")
        return {}

df = pd.read_csv(INPUT_TSV, sep='\t', dtype=str)
df = df[df['antigen_type'].str.contains('protein', na=False)]
df = df[(df['Hchain'].notna()) & (df['Hchain'] != 'NA') |
        (df['Lchain'].notna()) & (df['Lchain'] != 'NA')]

checkpoint_files = sorted(CHECKPOINT_DIR.glob("sabdab_h+l_checkpoint_*.fasta"),
                          key=lambda p: int(re.findall(r"\d+", p.stem)[-1]) if re.findall(r"\d+", p.stem) else 0)

fetched = []
processed_ids = set()
total_fetched = 0

if checkpoint_files:
    last_checkpoint = checkpoint_files[-1]
    print(f"Resuming from checkpoint: {last_checkpoint.name}")
    fetched = last_checkpoint.read_text().splitlines()
    for i in range(0, len(fetched), 2):
        header = fetched[i]
        if header.startswith(">"):
            id_pair = header[1:]
            processed_ids.add(id_pair)
    total_fetched = len(processed_ids)
else:
    print("Starting from scratch")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    pdb_id = row["pdb"].lower().strip()
    chain_seqs = get_fasta_chains(pdb_id)

    for chain_type in ['Hchain', 'Lchain']:
        chain_id = row.get(chain_type)
        if not isinstance(chain_id, str) or chain_id == 'NA':
            continue

        label = f"{pdb_id}-{chain_type[0]}-{chain_id}"
        if label in processed_ids:
            continue

        matched_chain_id = None
        if chain_id in chain_seqs:
            matched_chain_id = chain_id
        else:
            for candidate in chain_seqs:
                if candidate.lower() == chain_id.lower():
                    matched_chain_id = candidate
                    break

        if not matched_chain_id:
            tqdm.write(f"[✗] {label} — chain not found in PDB (even case-insensitive)")
            continue

        sequence = chain_seqs[matched_chain_id]
        if not sequence:
            continue

        fetched.append(f">{label}\n{sequence}")
        processed_ids.add(label)
        total_fetched += 1

        if total_fetched > 0 and total_fetched % CHECKPOINT_INTERVAL == 0:
            checkpoint_path = CHECKPOINT_DIR / f"sabdab_h+l_checkpoint_{total_fetched}.fasta"
            checkpoint_path.write_text("\n".join(fetched))
            tqdm.write(f"Checkpoint saved: {checkpoint_path}")

if fetched:
    Path(OUTPUT_FASTA).write_text("\n".join(fetched))
    print(f"\nSaved {len(fetched)//2} sequences to {OUTPUT_FASTA}")
else:
    print("\nNo sequences were fetched.")

In [ ]:
!cd-hit -i ../data/sabdab_h+l_chains_protein.fasta -o ../data/sabdab_clusters_h+l_protein.fasta -c 0.9 -n 5 -d 0 -T 4 -M 16000

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

clstr_path = "../data/sabdab_clusters_h+l_protein.fasta.clstr"

clusters = {}
with open(clstr_path) as f:
    cluster_id = None
    for line in f:
        line = line.strip()
        if line.startswith(">Cluster"):
            cluster_id = int(line.split()[1])
            clusters[cluster_id] = []
        else:
            parts = line.split(">")
            if len(parts) > 1:
                seq_id = parts[1].split("...")[0]
                clusters[cluster_id].append(seq_id)

G = nx.Graph()
color_map = {}
cluster_colors = {}

print("Building graph...")
for cl_id in tqdm(clusters):
    members = clusters[cl_id]
    if len(members) < 2:
        continue
    cluster_color = "#{:06x}".format(random.randint(0, 0xFFFFFF))
    cluster_colors[cl_id] = cluster_color
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            G.add_edge(members[i], members[j], cluster=cl_id)
        color_map[members[i]] = cluster_color

node_colors = [color_map.get(node, "#cccccc") for node in G.nodes]

print(f"Total nodes: {len(G.nodes)}, edges: {len(G.edges)}")

plt.figure(figsize=(18, 18))
print("Generating layout...")
pos = nx.spring_layout(G, seed=42, k=0.1)
print("Rendering graph...")
nx.draw(G, pos, node_color=node_colors, with_labels=False, node_size=30, edge_color="#333333", alpha=0.8)
plt.title("CD-HIT Clusters Colored by Group (All Clusters)")
plt.savefig("../data/cluster_data.svg", format="svg")
plt.show()


In [ ]:
from pathlib import Path
from Bio import SeqIO
import pandas as pd
import re

fasta_path = Path("../data/sabdab_h+l_chains_protein.fasta")
clstr_path = Path("../data/sabdab_clusters_h+l_protein.fasta.clstr")

seq_dict = {}
for record in SeqIO.parse(fasta_path, "fasta"):
    seq_dict[record.id] = str(record.seq)

clusters = []
current_cluster = None

with open(clstr_path) as f:
    for line in f:
        if line.startswith(">Cluster"):
            current_cluster = int(line.strip().split()[-1])
        else:
            match = re.search(r">(.+?)\.\.\.", line)
            if match:
                seq_id = match.group(1)
                clusters.append((seq_id, current_cluster))

data = []
for seq_id, cluster_id in clusters:
    sequence = seq_dict.get(seq_id)
    if not sequence:
        continue
    parts = seq_id.split("-")
    if len(parts) != 3:
        continue
    pdb, chain_type, chain_id = parts
    data.append({
        "pdb": pdb,
        "chain_type": chain_type,
        "chain_id": chain_id,
        "sequence": sequence,
        "cluster": cluster_id
    })

df = pd.DataFrame(data)
df.to_csv("../data/sabdab_h+l_clusters.csv", index=False)
print("Saved CSV with clusters to sabdab_h+l_clusters.csv")

In [ ]:


from Bio import SeqIO
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

fasta_path = "../data/sabdab_h+l_chains_protein.fasta"

seq_lengths = [len(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")]
length_series = pd.Series(seq_lengths)
stats = length_series.describe()

min_len = min(seq_lengths)
max_len = max(seq_lengths)
bins = np.arange(min_len, max_len + 2)

plt.figure(figsize=(14, 6))
plt.hist(seq_lengths, bins=bins, color='skyblue', edgecolor='black')
plt.axvline(stats["mean"], color='red', linestyle='dashed', linewidth=2, label=f"Mean: {stats['mean']:.1f}")
plt.axvline(stats["50%"], color='orange', linestyle='dashed', linewidth=2, label=f"Median: {stats['50%']:.1f}")
plt.title("Distribution of Protein Sequence Lengths")
plt.xlabel("Sequence Length")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
import random
from pathlib import Path
import numpy as np
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import AlignIO, SeqIO


input_fasta = Path("../data/sabdab_h+l_chains_protein.fasta").resolve()
output_csv = Path("../data/sabdab_h+l_cluster_combined_cdr.csv")
work_dir = Path("../data/mafft_tmp").resolve()
work_dir.mkdir(parents=True, exist_ok=True)
aligned_file = work_dir / "aligned_sample.fasta"
filtered_fasta = work_dir / "sampled_filtered.fasta"

sequences = list(SeqIO.parse(input_fasta, "fasta"))
print(f"Total sequences: {len(sequences)}")

lengths = np.array([len(rec.seq) for rec in sequences])
mean, std = lengths.mean(), lengths.std()
lower, upper = mean - std, mean + std
filtered = [rec for rec in sequences if lower <= len(rec.seq) <= upper]
print(f"Filtered sequences (mean ± std): {len(filtered)}")

random.seed(13)
n = min(1000, len(filtered))
sampled = random.sample(filtered, n)
SeqIO.write(sampled, filtered_fasta, "fasta")

print("Running MAFFT...")
process = subprocess.run(
    ["mafft", "--auto", str(filtered_fasta)],
    capture_output=True,
    text=True
)
if process.returncode != 0:
    print("MAFFT failed:", process.stderr)
    exit(1)
with open(aligned_file, "w") as out:
    out.write(process.stdout)
print("MAFFT alignment complete.")

alignment = AlignIO.read(aligned_file, "fasta")
length = alignment.get_alignment_length()
coverage = [sum(col != '-' for col in alignment[:, i]) for i in range(length)]
binary_matrix = np.array([[0 if aa == '-' else 1 for aa in str(rec.seq)] for rec in alignment])

fig_cov, ax_cov = plt.subplots(figsize=(20, 6))
ax_cov.plot(range(length), coverage, color='green')
ax_cov.set_title("Residue Coverage Across Alignment (MAFFT)")
ax_cov.set_xlabel("Alignment Position")
ax_cov.set_ylabel("Residues Present")
ax_cov.grid(True)
fig_cov.tight_layout()
fig_cov.savefig(work_dir / "alignment_coverage.svg", format="svg")
plt.show()

fig_heat, ax_heat = plt.subplots(figsize=(20, 6))
sns.heatmap(binary_matrix, cmap="Greens", cbar=True, xticklabels=50, yticklabels=False, ax=ax_heat)
ax_heat.set_title("Residue Presence Heatmap (MAFFT)")
ax_heat.set_xlabel("Alignment Position")
ax_heat.set_ylabel("Sequences")
fig_heat.tight_layout()
fig_heat.savefig(work_dir / "alignment_heatmap.svg", format="svg")
plt.show()

In [ ]:
import re
import requests
import pandas as pd
from pathlib import Path
from collections import Counter
from Bio import AlignIO
from Bio.Align import PairwiseAligner
from bs4 import BeautifulSoup
from tqdm import tqdm

alignment_file = Path("../data/mafft_tmp/aligned_sample.fasta")
output_csv = Path("../data/cdr_table.csv")

alignment = AlignIO.read(alignment_file, "fasta")

def top_kabat_variable_windows_numpy(alignment, min_len=5, max_len=15, top_n=3, max_gap_ratio=0.4):
    aln_length = alignment.get_alignment_length()
    n_seqs = len(alignment)
    seq_array = np.array([list(str(rec.seq)) for rec in alignment])
    gap_mask = (seq_array == "-").astype(np.uint8)

    variability_scores = np.zeros(aln_length)
    for i in range(aln_length):
        column = seq_array[:, i]
        counts = Counter(column)
        counts.pop("-", None)
        if counts:
            n = len(counts)
            f = counts.most_common(1)[0][1]
            variability_scores[i] = n / f if f > 0 else 0

    windows = []
    for w in range(min_len, max_len + 1):
        for i in range(aln_length - w + 1):
            gap_ratio = gap_mask[:, i:i+w].sum() / (w * n_seqs)
            if gap_ratio > max_gap_ratio:
                continue
            score = variability_scores[i:i + w].sum()
            windows.append((i, i + w, score))

    windows.sort(key=lambda x: x[2], reverse=True)
    selected = []
    for start, end, score in windows:
        if all(end <= s or start >= e for s, e, _ in selected):
            selected.append((start, end, score))
        if len(selected) >= top_n:
            break

    if len(selected) < top_n:
        fallback = []
        w = min_len
        for i in range(aln_length - w + 1):
            gap_ratio = gap_mask[:, i:i+w].sum() / (w * n_seqs)
            if gap_ratio > max_gap_ratio:
                continue
            score = variability_scores[i:i + w].sum()
            fallback.append((i, i + w, score))
        fallback.sort(key=lambda x: x[2], reverse=True)
        for start, end, score in fallback:
            if all(end <= s or start >= e for s, e, _ in selected):
                selected.append((start, end, score))
            if len(selected) >= top_n:
                break

    return selected[:top_n]

def fetch_real_cdr(pdb_id, chain_id, loop):
    base_url = "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/cdrviewer/"
    for cid in [chain_id, chain_id.lower()]:
        url = f"{base_url}?pdb={pdb_id}&loop={loop}&chain={cid}&CDRdef=chothia"
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
            table = soup.find("table")
            if not table:
                continue
            for row in table.find_all("tr"):
                if "Sequence" in row.text:
                    cells = row.find_all("td")
                    if len(cells) >= 2:
                        return cells[-1].text.strip()
        except:
            continue
    return ""

def classify_cdr_relation_smith_waterman(full_seq, kabat_cdrs, chothia_cdrs):
    aligner = PairwiseAligner()
    aligner.mode = "local"
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -1

    def sw_pos(subseq, seq):
        if not subseq or not seq:
            return None, None
        alignments = aligner.align(seq, subseq)
        if not alignments:
            return None, None
        aln = alignments[0]
        query_aln = aln.aligned[0]
        if len(query_aln) == 0:
            return None, None
        start = query_aln[0][0]
        end = query_aln[-1][1]
        return start, end

    relations = []
    for kabat_cdr, chothia_cdr in zip(kabat_cdrs, chothia_cdrs):
        k_start, k_end = sw_pos(kabat_cdr, full_seq)
        c_start, c_end = sw_pos(chothia_cdr, full_seq)

        if None in (k_start, c_start):
            relations.append("missing")
            continue

        k_center = (k_start + k_end) // 2
        c_center = (c_start + c_end) // 2
        dist = abs(k_center - c_center)

        if dist == 0:
            relations.append("overlap")
        elif dist <= 5:
            relations.append("near")
        elif dist <= 15:
            relations.append("far")
        else:
            relations.append("distant")
    return relations


windows = top_kabat_variable_windows_numpy(alignment)

records = []
for rec in tqdm(alignment, desc="Processing sequences"):
    match = re.match(r"([0-9a-z]{4})[-|_]+([hl])[-|_]+([A-Za-z0-9])", rec.id.lower())
    if not match:
        continue
    pdb_id, chain_type, chain_id = match.groups()
    pdb_id = pdb_id.lower()
    chain_type = chain_type.upper()
    chain_id = chain_id.upper()

    aln_seq = str(rec.seq)
    raw_seq = aln_seq.replace("-", "")
    kabat_like_cdrs = [aln_seq[start:end].replace("-", "") for start, end, _ in windows]
    while len(kabat_like_cdrs) < 3:
        kabat_like_cdrs.append("")

    real_cdrs = []
    for i in range(1, 4):
        loop = f"CDR{chain_type}{i}"
        real = fetch_real_cdr(pdb_id, chain_id, loop)
        real_cdrs.append(real)

    cdr_relations = classify_cdr_relation_smith_waterman(raw_seq, kabat_like_cdrs, real_cdrs)

    records.append({
        "pdb_id": pdb_id,
        "chain_type": chain_type,
        "chain_id": chain_id,
        "sequence": raw_seq,
        "kabat_like_cdr1": kabat_like_cdrs[0],
        "kabat_like_cdr2": kabat_like_cdrs[1],
        "kabat_like_cdr3": kabat_like_cdrs[2],
        "chothia_cdr1": real_cdrs[0],
        "chothia_cdr2": real_cdrs[1],
        "chothia_cdr3": real_cdrs[2],
        "cdr1_relation": cdr_relations[0],
        "cdr2_relation": cdr_relations[1],
        "cdr3_relation": cdr_relations[2],
    })

df = pd.DataFrame(records)
df.to_csv(output_csv, index=False)
print(df.head(10))

In [ ]:
import pandas as pd
from pathlib import Path
from Bio.PDB import PDBList
import pymol2

pdb_dir = Path("../data/pdb_files")
pdb_dir.mkdir(parents=True, exist_ok=True)

session_dir = Path("../data/protein10_pse")
session_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/cdr_table.csv")
filtered = df.dropna(subset=["chothia_cdr1", "kabat_like_cdr1"])
sampled = filtered.sample(n=10, random_state=13).reset_index(drop=True)

def fetch_pdb(pdb_id: str, dest_folder: Path) -> Path:
    pdbl = PDBList()
    pdb_file = dest_folder / f"{pdb_id}.pdb"
    if not pdb_file.exists():
        pdbl.retrieve_pdb_file(pdb_id, pdir=dest_folder, file_format="pdb")
        raw_file = dest_folder / f"pdb{pdb_id}.ent"
        if raw_file.exists():
            raw_file.rename(pdb_file)
    return pdb_file

for idx, row in sampled.iterrows():
    pdb_id = row["pdb_id"].lower()
    fetch_pdb(pdb_id, pdb_dir)

def find_subseq_positions(subseq, seq):
    idx = seq.find(subseq)
    return (idx + 1, idx + len(subseq)) if idx != -1 else (None, None)

for i, row in sampled.iterrows():
    pdb_id = row["pdb_id"].lower()
    chain = row["chain_id"]
    sequence = row["sequence"]
    pdb_file = pdb_dir / f"{pdb_id}.pdb"
    pse_file = session_dir / f"{pdb_id}_{chain}.pse"

    cdrs = {
        "kabat1": row["kabat_like_cdr1"],
        "kabat2": row["kabat_like_cdr2"],
        "kabat3": row["kabat_like_cdr3"],
        "chothia1": row["chothia_cdr1"],
        "chothia2": row["chothia_cdr2"],
        "chothia3": row["chothia_cdr3"],
    }

    color_map = {
        "kabat1": "blue",
        "kabat2": "blue",
        "kabat3": "blue",
        "chothia1": "red",
        "chothia2": "red",
        "chothia3": "red",
    }

    with pymol2.PyMOL() as pymol:
        pymol.cmd.reinitialize()
        pymol.cmd.load(str(pdb_file), "prot")
        pymol.cmd.hide("everything")
        pymol.cmd.show("cartoon", "prot")
        pymol.cmd.color("grey", "prot")

        pymol.cmd.set("label_size", 14)
        pymol.cmd.set("label_color", "black")

        for name, cdr in cdrs.items():
            if not isinstance(cdr, str) or len(cdr) < 3:
                continue
            start, end = find_subseq_positions(cdr, sequence)
            if start is None:
                continue
            sele = f"(chain {chain} and resi {start}-{end})"
            pymol.cmd.select(name, sele)
            pymol.cmd.color(color_map[name], name)
            pymol.cmd.show("sticks", name)

            try:
                center_resi = (start + end) // 2
                label_text = name.upper()
                pymol.cmd.label(f"(chain {chain} and resi {center_resi})", f'"{label_text}"')
            except:
                continue

        pymol.cmd.bg_color("white")
        pymol.cmd.orient()
        pymol.cmd.save(str(pse_file))

    print(f"Saved session: {pse_file}")

In [ ]:


import pandas as pd
import numpy as np
import random
from pathlib import Path
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import AlignIO
from scipy.spatial.distance import pdist, squareform

input_csv = Path("../data/sabdab_h+l_clusters.csv")
work_dir = Path("../data/mafft_random_cluster")
work_dir.mkdir(parents=True, exist_ok=True)
cluster_fasta = work_dir / "random_cluster.fasta"
aligned_fasta = work_dir / "aligned_random_cluster.fasta"

df = pd.read_csv(input_csv)
print(f"Total sequences: {len(df)}")

random.seed(13)
cluster_id = random.choice(df["cluster"].unique())
cluster_df = df[df["cluster"] == cluster_id].copy()
print(f"Selected cluster: {cluster_id}, n = {len(cluster_df)}")

with open(cluster_fasta, "w") as f:
    for _, row in cluster_df.iterrows():
        name = f"{row['pdb']}_{row['chain_id']}"
        seq = row["sequence"]
        if isinstance(seq, str) and len(seq) > 0:
            f.write(f">{name}\n{seq}\n")

print("Running MAFFT...")
result = subprocess.run(
    ["mafft", "--auto", str(cluster_fasta)],
    capture_output=True,
    text=True
)
if result.returncode != 0:
    raise RuntimeError("MAFFT failed:\n" + result.stderr)

with open(aligned_fasta, "w") as out:
    out.write(result.stdout)

print("MAFFT alignment complete.")

alignment = AlignIO.read(aligned_fasta, "fasta")
labels = [rec.id for rec in alignment]
length = alignment.get_alignment_length()

coverage = [sum(aa != '-' for aa in alignment[:, i]) for i in range(length)]

plt.figure(figsize=(16, 4))
plt.plot(range(length), coverage, color='green')
plt.title("Residue Coverage per Alignment Position")
plt.xlabel("Position")
plt.ylabel("Non-gap residues")
plt.grid(True)
plt.tight_layout()
plt.savefig(work_dir / "random_cluster_coverage.svg", format="svg")
plt.show()

binary_matrix = np.array([[0 if aa == '-' else 1 for aa in str(rec.seq)] for rec in alignment])

plt.figure(figsize=(16, 5))
sns.heatmap(binary_matrix, cmap="Greens", cbar=True, xticklabels=50, yticklabels=False)
plt.title("Residue Presence Heatmap")
plt.xlabel("Alignment Position")
plt.ylabel("Sequences")
plt.tight_layout()
plt.savefig(work_dir / "random_cluster_heatmap.svg", format="svg")
plt.show()

jaccard_dist = pdist(binary_matrix, metric="jaccard")
similarity_matrix = 1 - squareform(jaccard_dist)

g = sns.clustermap(
    similarity_matrix,
    cmap="viridis",
    figsize=(12, 12),
    metric="euclidean",
    method="average",
    xticklabels=labels,
    yticklabels=labels
)
g.ax_heatmap.set_title(f"Clustered Similarity Heatmap — Cluster {cluster_id}", fontsize=12)
g.savefig(work_dir / "random_similarity_clustermap.svg", format="svg")
plt.show()



In [ ]:
from pathlib import Path
from Bio import SeqIO
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

fasta_path = Path("../data/sabdab_h+l_chains_protein.fasta")
clstr_path = Path("../data/sabdab_clusters_h+l_protein.fasta.clstr")

seq_dict = {rec.id: str(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")}

clusters = []
current_cluster = None

with open(clstr_path) as f:
    for line in f:
        if line.startswith(">Cluster"):
            current_cluster = int(line.strip().split()[-1])
        else:
            match = re.search(r">(.+?)\.\.\.", line)
            if match:
                seq_id = match.group(1)
                clusters.append((seq_id, current_cluster))

data = []
for seq_id, cluster_id in clusters:
    sequence = seq_dict.get(seq_id)
    if not sequence:
        continue
    parts = seq_id.split("-")
    if len(parts) != 3:
        continue
    pdb, chain_type, chain_id = parts
    data.append({
        "pdb": pdb.lower(),
        "chain_type": chain_type.upper(),
        "chain_id": chain_id.upper(),
        "sequence": sequence,
        "cluster": cluster_id
    })

df = pd.DataFrame(data)
print(f"Total parsed entries: {len(df)}")

def fetch_chothia_cdr(pdb_id, chain_id, loop):
    base_url = "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/cdrviewer/"
    for cid in [chain_id, chain_id.lower()]:
        url = f"{base_url}?pdb={pdb_id}&loop={loop}&chain={cid}&CDRdef=chothia"
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
            table = soup.find("table")
            if not table:
                continue
            for row in table.find_all("tr"):
                if "Sequence" in row.text:
                    cells = row.find_all("td")
                    if len(cells) >= 2:
                        return cells[-1].text.strip()
        except:
            continue
    return ""

records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Fetching CDRs"):
    pdb = row["pdb"]
    chain_id = row["chain_id"]
    chain_type = row["chain_type"]
    cluster = row["cluster"]

    cdrs = []
    for i in range(1, 4):
        loop = f"CDR{chain_type}{i}"
        cdr = fetch_chothia_cdr(pdb, chain_id, loop)
        cdrs.append(cdr)

    records.append({
        "cluster": cluster,
        "cdr1": cdrs[0],
        "cdr2": cdrs[1],
        "cdr3": cdrs[2],
    })

cdr_df = pd.DataFrame(records)
cdr_df = cdr_df[(cdr_df["cdr1"] != "") & (cdr_df["cdr2"] != "") & (cdr_df["cdr3"] != "")]

cdr_df.to_csv("../data/cdr_by_cluster.csv", index=False, columns=["cluster", "cdr1", "cdr2", "cdr3"])
print(f"Saved {len(cdr_df)} pdbs to cdr_by_cluster.csv")